In [1]:
import json
import os
import pandas as pd

def load_json_data(file_path):
    """Reads and returns data from a JSON file."""
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found in the current directory.")
        return None

    with open(file_path, 'r') as file:
        return json.load(file)

def inspect_csv_data(file_path):
    df = pd.read_csv(file_path)
    num_data, num_features = df.shape
    return num_data, num_features

def compute_num_params_nn(all_layers, layer_in, layer_out):
    total_params = 0
    total_layers = [layer_in] + all_layers + [layer_out]
    for i in range(len(total_layers) - 1):
        n_in = total_layers[i]
        n_out = total_layers[i+1]

        # Calculation formula: (Inputs * Outputs) + Outputs
        weights = n_in * n_out
        biases = n_out
        subtotal = weights + biases
        total_params += subtotal
    return total_params

In [5]:
"""Extracts and prints key metrics from the specified JSON files."""
# Define file names
name_list = [
    # 'CrossedBarrel',
    'AgNP',
    # 'P3HT',
    'CO2AR4500',
    'CO2AH4500',
    'CO2RRLCA',
    'CO2RRNPV',
    # 'CO2RRMSP',
    'CO2HPx10',
]

data_performance = {'name': [], 'model': [], 'R2': [], 'num_params': [], 'num_features': [], 'num_datapoints': []}

data_params_mlp = []
data_params_kan = []
dir_project = r"D:\pykan\github\workflows\Hyein"

for name in name_list:
    mlp_file = os.path.join("material_nn_models", name, f"{name}_metrics.json")
    kan_file = os.path.join("material_kan_models", name, f"{name}_kan_metrics.json")
    data_path = os.path.join(dir_project, 'data', f"{name}.csv")

    # Load data
    mlp_data = load_json_data(mlp_file)
    kan_data = load_json_data(kan_file)

    n_data, n_cols = inspect_csv_data(data_path)
    n_in = n_cols - 1
    n_out = 1

    # Process Standard Neural Network Data
    # if mlp_data:
    #     mlp_params = mlp_data.get('best_params')
    #     nn_layers = mlp_params.get("hidden_layer_sizes")
    #     nn_params = compute_num_params_nn(nn_layers, n_in, n_out)
    #
    #     data_performance['name'].append(name)
    #     data_performance['num_features'].append(n_cols)
    #     data_performance['num_datapoints'].append(n_data)
    #     data_performance['model'].append('NN')
    #     data_performance['R2'].append(mlp_data.get('test_r2', 0))
    #     data_performance['num_params'].append(nn_params)
    #
    #     data_params_mlp.append({
    #         'name': name,
    #         'R2': mlp_data.get('test_r2', 0),
    #         'num_params': nn_params,
    #         'hidden_layer_sizes': nn_layers,
    #         'alpha': mlp_params.get('alpha', 0),
    #         'learning_rate_init': mlp_params.get('learning_rate_init', 0),
    #     })

    # Process KAN / Symbolic Regression Data
    if kan_data:
        kan_params = kan_data.get('best_params')
        kan_num_params = n_in * n_out * (kan_params['grid'] + kan_params['k'] + 1)

        data_performance['name'].append(name)
        data_performance['num_features'].append(n_cols)
        data_performance['num_datapoints'].append(n_data)
        data_performance['model'].append('KAN')
        data_performance['R2'].append(kan_data.get('test_r2', 0))
        data_performance['num_params'].append(kan_num_params)

        data_params_kan.append({
            'name': name,
            'R2': kan_data.get('test_r2', 0),
            'num_params': kan_num_params,
            'layers': kan_params.get('n_layers', 0),
            'grid': kan_params.get('grid', 0),
            'steps': kan_params.get('steps', 0),
            'lamb': kan_params.get('lamb', 0),
            'lamb_coef': kan_params.get('lamb_coef', 0),
            'lamb_coefdiff': kan_params.get('lamb_coefdiff', 0),
            'lamb_entropy': kan_params.get('lamb_entropy', 0),
            'lr': kan_params.get('lr', 0),
            'sym_range': kan_params.get('sym_range', 0),
        })

df_performance = pd.DataFrame(data_performance)
df_performance.to_csv(os.path.join(dir_project, 'figures_for_paper', 'performance.csv'))
print(df_performance)

df_params_mlp = pd.DataFrame(data_params_mlp)
df_params_mlp.to_csv(os.path.join(dir_project, 'figures_for_paper', 'params_mlp.csv'))
df_params_kan = pd.DataFrame(data_params_kan)
df_params_kan.to_csv(os.path.join(dir_project, 'figures_for_paper', 'params_kan.csv'))
print(df_params_mlp)
print(df_params_kan)

Error: File 'material_nn_models\CO2AR4500\CO2AR4500_metrics.json' not found in the current directory.
Error: File 'material_nn_models\CO2AH4500\CO2AH4500_metrics.json' not found in the current directory.
        name model        R2  num_params  num_features  num_datapoints
0       AgNP   KAN  0.812329          70             6            3295
1  CO2AR4500   KAN  0.761509          54             7            5394
2  CO2AH4500   KAN  0.761497          54             7            5394
3   CO2RRLCA   KAN  0.965041          72             9            3677
4   CO2RRNPV   KAN  0.971202         112             9            3677
5   CO2HPx10   KAN  0.858984         144            17            6949
Empty DataFrame
Columns: []
Index: []
        name        R2  num_params  layers  grid  steps  lamb  lamb_coef   
0       AgNP  0.812329          70       1    10     20  0.01       0.01  \
1  CO2AR4500  0.761509          54       1     5     50  0.01       1.00   
2  CO2AH4500  0.761497          5

In [14]:
name = 'CO2HPx10'
seed_list = [i for i in range(10)]

data_performance = {'name': [], 'seed': [], 'model': [], 'R2': [], 'num_params': []}
for seed in seed_list:
    dir_project = r"D:\pykan\github\workflows\Hyein"
    mlp_file = os.path.join("material_nn_models", name+f"_seed_{seed}", f"{name}_metrics.json")
    kan_file = os.path.join("material_kan_models", name+f"_seed_{seed}", f"{name}_kan_metrics.json")
    data_path = os.path.join(dir_project, 'data', f"{name}.csv")

    # Load data
    mlp_data = load_json_data(mlp_file)
    kan_data = load_json_data(kan_file)

    n_cols = inspect_csv_data(data_path)
    n_in = n_cols - 1
    n_out = 1

    # Process Standard Neural Network Data
    if mlp_data:
        nn_layers = mlp_data.get('best_params').get("hidden_layer_sizes")
        print(nn_layers)
        nn_params = compute_num_params_nn(nn_layers, n_in, n_out)

        data_performance['name'].append(name)
        data_performance['seed'].append(seed)
        data_performance['model'].append('NN')
        data_performance['R2'].append(mlp_data.get('test_r2', 0))
        data_performance['num_params'].append(nn_params)

    # Process KAN / Symbolic Regression Data
    if kan_data:
        kan_params = kan_data.get('best_params')
        kan_params = n_in * n_out * (kan_params['grid'] + kan_params['k'] + 1)

        data_performance['name'].append(name)
        data_performance['seed'].append(seed)
        data_performance['model'].append('KAN')
        data_performance['R2'].append(kan_data.get('test_r2', 0))
        data_performance['num_params'].append(kan_params)
df_performance = pd.DataFrame(data_performance)
print(df_performance)

       name  seed model        R2  num_params
0  CO2HPx10     0    NN  0.834723        3713
1  CO2HPx10     1    NN  0.835082        3713
2  CO2HPx10     2    NN  0.844408        3985
3  CO2HPx10     3    NN  0.840174        3985
4  CO2HPx10     4    NN  0.839968        3985
5  CO2HPx10     5    NN  0.840036        3985
6  CO2HPx10     6    NN  0.827737       10497
7  CO2HPx10     7    NN  0.840024        3985
8  CO2HPx10     8    NN  0.828249       10497
9  CO2HPx10     9    NN  0.840271        3985


In [4]:
# ============================================================
# KAN Hyperparameters for Analytic Functions
# ============================================================
import yaml

def _yaml_tuple_constructor(loader, node):
    return tuple(loader.construct_sequence(node))
yaml.add_constructor('tag:yaml.org,2002:python/tuple', _yaml_tuple_constructor, Loader=yaml.SafeLoader)

analytical_results_dir = os.path.join(dir_project, 'analytical_results')

rows = []
for func_name in sorted(os.listdir(analytical_results_dir)):
    metrics_path = os.path.join(analytical_results_dir, func_name, 'kan_models', f'{func_name}_kan_metrics.json')
    config_path  = os.path.join(analytical_results_dir, func_name, 'kan_models', f'{func_name}_best_kan_model_config.yml')

    if not os.path.exists(metrics_path):
        continue

    with open(metrics_path) as f:
        metrics = json.load(f)
    p = metrics.get('best_params', {})

    # Architecture from config YAML (width field)
    arch_str  = None
    num_params = None
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = yaml.safe_load(f)
        width = cfg.get('width', [])
        if width:
            arch_str = ' → '.join(str(w[0]) for w in width)
            grid, k = p.get('grid'), p.get('k')
            if None not in (grid, k):
                num_params = sum(
                    width[i][0] * width[i + 1][0] * (grid + k + 1)
                    for i in range(len(width) - 1)
                )

    rows.append({
        'func':         func_name,
        'test_R2':      round(metrics.get('test_r2', float('nan')), 4),
        'architecture': arch_str,
        'n_layers':     p.get('n_layers'),
        'grid':         p.get('grid'),
        'k':            p.get('k'),
        'steps':        p.get('steps'),
        'lr':           p.get('lr'),
        'lamb':         p.get('lamb'),
        'lamb_coef':    p.get('lamb_coef'),
        'lamb_entropy': p.get('lamb_entropy'),
        'num_params':   num_params,
    })

df_analytical_kan = pd.DataFrame(rows)
save_path = os.path.join(dir_project, 'figures_for_paper', 'params_kan_analytical.csv')
df_analytical_kan.to_csv(save_path, index=False)
print(f"Saved → {save_path}")
# display(df_analytical_kan)


Saved → D:\pykan\github\workflows\Hyein\figures_for_paper\params_kan_analytical.csv


In [5]:
# Choose a subset of analytic functions to display
# Set to None to show all
selected_funcs = [
    'log2',
    'logarithm',
    'exponential',
    # 'multiplication',
    # 'convolution',
    'rosenbrock',
    # 'log_sum_2d',
    # 'log_sum_5d',
    # 'log_sum_10d',
    # 'log_sum_30d',
]

if selected_funcs is not None:
    df_selected = df_analytical_kan[df_analytical_kan['func'].isin(selected_funcs)].copy()
    df_selected = df_selected.set_index('func').loc[[f for f in selected_funcs if f in df_selected['func'].values]]
    df_selected = df_selected.reset_index()
else:
    df_selected = df_analytical_kan.copy()

display(df_selected)


,func,test_R2,architecture,n_layers,grid,k,steps,lr,lamb,lamb_coef,lamb_entropy,num_params
0,log2,0.9999,2 → 2 → 1,2,10,3,50,1.0,0.00,0.01,5.0,84
1,logarithm,1.0000,2 → 1,1,10,3,50,1.0,0.00,0.01,4.0,28
2,exponential,0.9995,2 → 1,1,10,3,20,0.1,0.01,1.00,0.1,28
3,rosenbrock,0.9106,2 → 2 → 1,2,10,3,50,1.0,0.00,0.00,1.0,84
